# Notebook 12: Residual Learning & MNIST MLP Project

**Module:** 05 Deep Learning  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand skip connections and gradient flow
2. Build complete PyTorch MLP on MNIST
3. Reimplement Day - 28 architecture in PyTorch
4. Complete Module 05 capstone

## 1. Residual Connections (He et al., 2016)

$$y = F(x) + x$$

Instead of learning $H(x)$, learn residual $F(x) = H(x) - x$.

**Why:** Gradients can flow directly through skip connection → train very deep networks.

Preview of Module 06 (ResNet) and water-bodies UNet++ skip connections.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
    def forward(self, x):
        return F.relu(self.fc(x) + x)  # skip connection

block = ResidualBlock(64)
x = torch.randn(16, 64)
print(f'Input shape: {x.shape}, Output shape: {block(x).shape}')

## 2. Complete MNIST MLP (PyTorch)

Full training loop — the template for all future modules.

In [ ]:
# Load MNIST
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data.astype(np.float32) / 255.0, mnist.target.astype(int)

# Subset for speed
X, y = X[:10000], y[:10000]
X_train, X_test = X[:8000], X[8000:]
y_train, y_test = y[:8000], y[8000:]

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

In [ ]:
class MNISTMLP(nn.Module):
    """Day - 28 equivalent in PyTorch: 784 → 256 → 128 → 10"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

model = MNISTMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    total_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    model.eval()
    with torch.no_grad():
        acc = (model(X_test_t.to(device)).argmax(1) == y_test_t.to(device)).float().mean()
    print(f'Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, test_acc={acc:.4f}')

## 3. Day - 28 Keras → PyTorch Mapping

| Keras | PyTorch |
|-------|--------|
| `Sequential()` | `nn.Sequential()` or `nn.Module` |
| `Dense(12, activation='relu')` | `nn.Linear(8, 12)` + `nn.ReLU()` |
| `model.compile(loss='binary_crossentropy')` | `F.binary_cross_entropy_with_logits` |
| `model.fit(x, y, epochs=5)` | Training loop with DataLoader |
| `model.save_weights('model.h5')` | `torch.save(model.state_dict(), 'model.pt')` |

## Module 05 Complete

You can now build, train, and debug neural networks in PyTorch.

**Next:** Module 06 CNN — convolutions, LeNet, ResNet.

## Exercise

Add a residual block to the MNIST MLP hidden layer. Does accuracy improve?

In [ ]:
# YOUR CODE HERE


---

## Summary

Residual learning enables deep networks. MNIST MLP = your first complete PyTorch training pipeline.